# Lab 4 — Chatbot RAG avec Graphe de Connaissances

**Prérequis :** Lab 2 (kb_expanded.ttl) + Ollama lancé (`ollama serve`) + modèle (`ollama pull llama3.2:1b`)

On va construire un chatbot qui génère des requêtes SPARQL à partir de questions en langage naturel, les exécute sur notre KB et retourne des réponses en français.

In [ ]:
import os, re, time, random, warnings
from collections import Counter
from typing import List, Tuple, Dict
from urllib.parse import unquote

import requests
import pandas as pd
import numpy as np
import gradio as gr
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL

warnings.filterwarnings("ignore")
print("imports ok")


In [ ]:
# LAB 4 — CHATBOT RAG AVEC GRAPHE DE CONNAISSANCES
# Auteurs : Bilal Gougis & Chadi Al Kerdi
# Cours   : Web Mining & Sémantique

import os, re, time
import requests
import pandas as pd
from typing import List, Tuple, Dict
from rdflib import Graph
from rdflib.namespace import RDF, RDFS, OWL

# CONFIGURATION

OLLAMA_URL   = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "mistral:7b"  # ← Modèle principal : bien meilleur en SPARQL

OCT_NS = "http://octopusbiology.lab/ontology/"
ENT_NS = "http://octopusbiology.lab/entity/"
WDT_NS = "http://www.wikidata.org/prop/direct/"
WD_NS  = "http://www.wikidata.org/entity/"

MAX_PREDICATES   = 80
MAX_CLASSES      = 40
SAMPLE_TRIPLES   = 30    # ← Plus d'exemples pour mistral:7b (window plus large)
MAX_RESULTS_SHOW = 20

EVAL_QUESTIONS = [
    "Quel est le poids maximum d'Enteroctopus dofleini ?",
    "Dans quels habitats vit Octopus vulgaris ?",
    "Quelles espèces de poulpes vivent dans l'Indo-Pacifique ?",
    "Quelle espèce de poulpe est mortelle pour l'humain ?",
    "Quel est le parent taxonomique d'Octopus vulgaris ?",
    "Quels sont les prédateurs des poulpes ?",
    "Quelles espèces de céphalopodes utilisent le camouflage ?",
    "À quel ordre appartient le genre Octopus ?",
]

print("✓ Configuration Lab 4 chargée")
print(f"  Modèle : {OLLAMA_MODEL}")


## Etape 1 : Chargement du graphe

In [ ]:

# ÉTAPE 1 — CHARGEMENT DU GRAPHE

def load_graph(ttl="kb_expanded.ttl", nt="kb_expanded.nt"):
    g = Graph()
    if os.path.exists(ttl):
        print(f"Chargement depuis {ttl}...")
        g.parse(ttl, format="turtle")
    elif os.path.exists(nt):
        print(f"Chargement depuis {nt}...")
        g.parse(nt, format="nt")
    else:
        raise FileNotFoundError("kb_expanded.ttl introuvable — exécutez Lab 2 d'abord.")
    print(f"✓ {len(g):,} triplets chargés")
    return g

try:
    g_rag = load_graph()
except FileNotFoundError as e:
    print(f"⚠ {e}")
    g_rag = Graph()

## Etape 2 : Résumé de schéma

In [ ]:


# ÉTAPE 2 — RÉSUMÉ DE SCHÉMA INTELLIGENT (adapté pour le LLM)

def shorten_uri(uri):
    for prefix, ns in [("oct", OCT_NS), ("ent", ENT_NS),
                       ("wdt", WDT_NS), ("wd", WD_NS),
                       ("rdfs", str(RDFS)), ("rdf", str(RDF))]:
        if uri.startswith(ns):
            return f"{prefix}:{uri[len(ns):]}"
    return uri

def get_entity_labels(g, limit=200):
    # Récupère les labels des entités pour les injecter dans le schéma.
    labels = {}
    for r in g.query(f# SELECT ?s ?label WHERE {{
    # Préfixes
    prefixes = "\n".join([
        f'PREFIX ent: <{ENT_NS}>',
        f'PREFIX oct: <{OCT_NS}>',
        f'PREFIX wdt: <{WDT_NS}>',
        f'PREFIX wd:  <{WD_NS}>',
        'PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>',
        'PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>',
        'PREFIX owl:  <http://www.w3.org/2002/07/owl#>',
    ])
    
    # Prédicats avec comptage
    pred_query = # SELECT ?p (COUNT(*) AS ?count) WHERE {
    preds = []
    for r in g.query(pred_query):
        short = shorten_uri(str(r.p))
        count = int(r['count'])
        preds.append(f"- {short} ({count} triplets)")
    
    # Classes
    clss = [shorten_uri(str(r.cls))
            for r in g.query(f"SELECT DISTINCT ?cls WHERE {{ ?s a ?cls }} LIMIT {MAX_CLASSES}")]
    
    # Labels des entités clés
    labels = get_entity_labels(g)
    label_lines = [f"- {uri} → \"{label}\"" for uri, label in list(labels.items())[:50]]
    
    # Exemples de triplets pertinents
    samples = get_relevant_samples(g)
    
    summary = f# {prefixes}
    return summary.strip()

print("Construction du résumé de schéma (version optimisée)...")
schema_summary = build_schema_summary(g_rag)
print(f"✓ Résumé : {len(schema_summary)} caractères")
print(f"\nAperçu (500 premiers chars):\n{schema_summary[:500]}")


## Etape 3 : Connexion Ollama

In [ ]:


# ÉTAPE 3 — CONNEXION OLLAMA

def check_ollama():
    try:
        if requests.get("http://localhost:11434", timeout=3).status_code == 200:
            print("✓ Ollama en cours d'exécution")
            return True
    except Exception:
        pass
    print("✗ Ollama inaccessible — lancez : ollama serve")
    return False

def ask_local_llm(prompt, model=OLLAMA_MODEL, timeout=180):
    # Appel au LLM local via Ollama. Timeout augmenté pour mistral:7b.
    try:
        r = requests.post(OLLAMA_URL,
                          json={
                              "model": model,
                              "prompt": prompt,
                              "stream": False,
                              "options": {
                                  "temperature": 0.1,      # Très déterministe pour le SPARQL
                                  "num_predict": 512,       # Limiter la longueur de sortie
                                  "top_p": 0.9,
                              }
                          },
                          timeout=timeout)
        if r.status_code != 200:
            raise RuntimeError(f"Erreur Ollama {r.status_code}")
        return r.json().get("response", "").strip()
    except requests.exceptions.ConnectionError:
        return "[ERREUR] Ollama non disponible. Lancez : ollama serve"
    except requests.exceptions.Timeout:
        return "[ERREUR] Timeout — le modèle met trop de temps"

ollama_ok = check_ollama()
if ollama_ok:
    print(f"\nTest avec {OLLAMA_MODEL}...")
    test = ask_local_llm("En une phrase : qu'est-ce qu'un poulpe ?")
    print(f"Test : {test[:300]}")
    
    # Vérifier que le modèle est bien chargé
    if "[ERREUR]" in test:
        print(f"\n⚠ Le modèle {OLLAMA_MODEL} n'est pas installé.")
        print(f"  Exécutez : ollama pull {OLLAMA_MODEL}")


## Etape 4 : Baseline (LLM seul sans KB)

In [ ]:


# ÉTAPE 4 — BASELINE (LLM sans KB)

def answer_no_rag(question, model=OLLAMA_MODEL):
    return ask_local_llm(
        f"Tu es un expert en biologie marine. Réponds de façon précise et concise.\n\n"
        f"Question : {question}\nRéponse :",
        model=model
    )

print("\n" + "=" * 60)
print("  BASELINE — Réponses LLM sans KB")
print("=" * 60)

baseline_results = {}
for q in EVAL_QUESTIONS:
    print(f"\n❓ {q}")
    baseline_results[q] = answer_no_rag(q)
    print(f"🤖 {baseline_results[q][:200]}")


## Etape 5 : Pipeline RAG (SPARQL + self-repair)

In [ ]:


# ÉTAPE 5 — PIPELINE RAG v3 (optimisé pour mistral:7b)

CODE_BLOCK_RE  = re.compile(r"```(?:sparql)?\s*(.*?)```", re.IGNORECASE | re.DOTALL)
MOTS_INTERDITS = ["SERVICE", "CONSTRUCT", "DESCRIBE", "INSERT", "DELETE", "LOAD", "CLEAR"]


def extract_sparql(text):
    # Extrait la requête SPARQL du texte LLM, en nettoyant le bruit.
    if not text or text.startswith("[ERREUR]"):
        return ""
    
    # 1. Chercher dans un bloc ```sparql ... ```
    m = CODE_BLOCK_RE.search(text)
    if m:
        sparql = m.group(1).strip()
    else:
        # 2. Chercher les lignes SPARQL dans le texte brut
        lines = text.split("\n")
        sparql_lines = []
        capture = False
        for line in lines:
            stripped = line.strip()
            upper = stripped.upper()
            if upper.startswith(("PREFIX", "SELECT")):
                capture = True
            if capture:
                sparql_lines.append(line)
                # Arrêter après la dernière accolade fermante
                if "}" in stripped and not upper.startswith("PREFIX"):
                    # Vérifier si les accolades sont équilibrées
                    full = "\n".join(sparql_lines)
                    if full.count("{") <= full.count("}"):
                        break
        sparql = "\n".join(sparql_lines).strip() if sparql_lines else ""
    
    if not sparql:
        return ""
    
    # 3. Nettoyer
    sparql = re.sub(r"```(?:sparql)?|```", "", sparql, flags=re.IGNORECASE).strip()
    
    # Couper après la dernière accolade fermante
    last_brace = sparql.rfind("}")
    if last_brace != -1:
        sparql = sparql[:last_brace + 1]
    
    # 4. Vérifier
    if "SELECT" not in sparql.upper():
        return ""
    
    return sparql


SPARQL_TEMPLATES = {
    "habitat": (
        f"PREFIX ent: <{ENT_NS}>\n"
        f"PREFIX oct: <{OCT_NS}>\n"
        f"PREFIX wdt: <{WDT_NS}>\n"
        "PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n"
        "SELECT ?espece ?label ?habitat ?habitat_label WHERE {{\n"
        "  ?espece rdfs:label ?label .\n"
        "  {{ ?espece oct:vivre ?habitat }} UNION {{ ?espece oct:habiter ?habitat }} "
        "UNION {{ ?espece wdt:P551 ?habitat }}\n"
        "  OPTIONAL {{ ?habitat rdfs:label ?habitat_label }}\n"
        '  FILTER(CONTAINS(LCASE(STR(?label)), "{kw}"))\n'
        "}} LIMIT 20"
    ),
    "taxonomie": (
        f"PREFIX ent: <{ENT_NS}>\n"
        f"PREFIX wdt: <{WDT_NS}>\n"
        f"PREFIX wd: <{WD_NS}>\n"
        "PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n"
        "SELECT ?espece ?label ?parent ?parent_label WHERE {{\n"
        "  ?espece rdfs:label ?label .\n"
        "  ?espece wdt:P171 ?parent .\n"
        "  OPTIONAL {{ ?parent rdfs:label ?parent_label }}\n"
        '  FILTER(CONTAINS(LCASE(STR(?label)), "{kw}"))\n'
        "}} LIMIT 20"
    ),
    "especes": (
        f"PREFIX ent: <{ENT_NS}>\n"
        f"PREFIX oct: <{OCT_NS}>\n"
        "PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\n"
        "PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n"
        "SELECT ?espece ?label WHERE {{\n"
        "  ?espece rdf:type oct:Species .\n"
        "  ?espece rdfs:label ?label .\n"
        '  FILTER(CONTAINS(LCASE(STR(?label)), "{kw}"))\n'
        "}} LIMIT 20"
    ),
    "proprietes": (
        f"PREFIX ent: <{ENT_NS}>\n"
        f"PREFIX oct: <{OCT_NS}>\n"
        "PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n"
        "SELECT ?propriete ?valeur WHERE {{\n"
        '  ?s rdfs:label ?label . FILTER(CONTAINS(LCASE(STR(?label)), "{kw}"))\n'
        "  ?s ?propriete ?valeur .\n"
        "  FILTER(?propriete != rdfs:label && ?propriete != rdfs:comment)\n"
        "}} LIMIT 30"
    ),
}

def detect_question_pattern(question):
    # Détecte le type de question pour choisir un template SPARQL.
    q = question.lower()
    
    if any(w in q for w in ["habitat", "vit", "vivre", "trouve", "répartition", "océan", "mer", "pacifique", "atlantique", "indo"]):
        return "habitat"
    if any(w in q for w in ["taxonom", "parent", "classif", "ordre", "famille", "genre", "appartient", "classe"]):
        return "taxonomie"
    if any(w in q for w in ["espèce", "species", "quels poulpes", "quelles pieuvres", "liste"]):
        return "especes"
    
    return None

def extract_keyword(question):
    # Extraire le mot-clé principal (nom d'espèce ou concept) de la question.
    q = question.lower()
    
    # Noms d'espèces connus
    SPECIES_NAMES = [
        "enteroctopus dofleini", "octopus vulgaris", "octopus cyanea",
        "hapalochlaena", "eledone cirrhosa", "amphioctopus marginatus",
        "callistoctopus macropus", "tremoctopus", "wunderpus photogenicus",
    ]
    for sp in SPECIES_NAMES:
        if sp in q:
            return sp
    
    # Noms communs
    COMMON_NAMES = {
        "pieuvre géante": "enteroctopus",
        "poulpe commun": "octopus vulgaris", 
        "pieuvre commune": "octopus vulgaris",
        "pieuvre à anneaux bleus": "hapalochlaena",
        "blue-ringed": "hapalochlaena",
        "pieuvre mimétique": "thaumoctopus",
    }
    for name, kw in COMMON_NAMES.items():
        if name in q:
            return kw
    
    # Mots-clés géographiques
    for geo in ["pacifique", "atlantique", "indo-pacifique", "méditerranée", "australie"]:
        if geo in q:
            return geo
    
    # Fallback : "poulpe" ou "pieuvre" ou "octopus"
    for default in ["poulpe", "pieuvre", "octopus", "céphalopode"]:
        if default in q:
            return default
    
    return "poulpe"


def generate_sparql(question, schema, model=OLLAMA_MODEL):
    # Génère une requête SPARQL — avec templates + LLM comme fallback.
    
    kw = extract_keyword(question)
    pattern = detect_question_pattern(question)
    
    # 1. Essayer un template si le pattern est reconnu
    if pattern and pattern in SPARQL_TEMPLATES:
        template_query = SPARQL_TEMPLATES[pattern].format(kw=kw)
        print(f"  → Pattern détecté : {pattern} (kw={kw})")
        return template_query
    
    # 2. Sinon, demander au LLM
    print(f"  → Pas de template, génération LLM (kw={kw})...")
    
    # Adapter le schéma à la taille du contexte du modèle
    limit = 5000 if "7b" in model or "mistral" in model else \
            3000 if "3b" in model else 1500
    schema_court = schema[:limit]
    
    prompt = f# Tu es un expert SPARQL. Génère une requête SPARQL 1.1 SELECT pour interroger un graphe RDF sur les poulpes/céphalopodes.
    
    raw_response = ask_local_llm(prompt, model=model)
    sparql = extract_sparql(raw_response)
    
    if not sparql:
        print("  ⚠ SPARQL vide → fallback propriétés")
        return SPARQL_TEMPLATES["proprietes"].format(kw=kw)
    
    if any(k in sparql.upper() for k in MOTS_INTERDITS):
        print("  ⚠ Mot interdit → fallback propriétés")
        return SPARQL_TEMPLATES["proprietes"].format(kw=kw)
    
    # Ajouter prefixes si manquants
    if "PREFIX" not in sparql.upper():
        prefixes = (
            f"PREFIX ent: <{ENT_NS}>\n"
            f"PREFIX oct: <{OCT_NS}>\n"
            f"PREFIX wdt: <{WDT_NS}>\n"
            f"PREFIX wd: <{WD_NS}>\n"
            "PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n"
            "PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\n"
        )
        sparql = prefixes + sparql
    
    # Ajouter LIMIT si absent
    if "LIMIT" not in sparql.upper():
        sparql = sparql.rstrip().rstrip("}") + "} LIMIT 20"
    
    return sparql


def repair_sparql(schema, question, bad_query, error_msg, model=OLLAMA_MODEL):
    # Répare une requête SPARQL invalide.
    schema_court = schema[:2000]
    prompt = f# La requête SPARQL suivante a échoué. Corrige-la.
    
    result = extract_sparql(ask_local_llm(prompt, model=model))
    if result:
        return result
    
    # Fallback intelligent basé sur la question
    kw = extract_keyword(question)
    return SPARQL_TEMPLATES["proprietes"].format(kw=kw)


def run_sparql_on_graph(g, query):
    # Exécute une requête SPARQL sur le graphe RDF.
    res   = g.query(query)
    vars_ = [str(v) for v in res.vars] if res.vars else []
    rows  = [tuple(str(c) if c is not None else "" for c in r) for r in res]
    return vars_, rows


def answer_with_sparql_rag(g, schema, question, try_repair=True, model=OLLAMA_MODEL):
    # Pipeline complet : template/génération SPARQL → exécution → repair → fallback.
    sparql = generate_sparql(question, schema, model=model)
    
    if not sparql or sparql.startswith("[ERREUR]"):
        kw = extract_keyword(question)
        sparql = SPARQL_TEMPLATES["proprietes"].format(kw=kw)
    
    # Tentative 1 : exécution directe
    try:
        v, r = run_sparql_on_graph(g, sparql)
        if r:  # Si on a des résultats
            return {"query": sparql, "vars": v, "rows": r, "repaired": False, "error": None}
        # Si 0 résultats, essayer un fallback plus large
        print("  ⚠ 0 résultats, tentative élargie...")
        kw = extract_keyword(question)
        fallback = SPARQL_TEMPLATES["proprietes"].format(kw=kw)
        v2, r2 = run_sparql_on_graph(g, fallback)
        return {"query": fallback, "vars": v2, "rows": r2, "repaired": False, "error": None}
    except Exception as e:
        err = str(e)
    
    # Tentative 2 : self-repair
    if try_repair:
        print(f"  ⚠ {err[:80]} — self-repair...")
        rep = repair_sparql(schema, question, sparql, err, model=model)
        try:
            v, r = run_sparql_on_graph(g, rep)
            return {"query": rep, "vars": v, "rows": r, "repaired": True, "error": None}
        except Exception as e2:
            pass
    
    # Tentative 3 : fallback garanti
    kw = extract_keyword(question)
    try:
        fallback = SPARQL_TEMPLATES["proprietes"].format(kw=kw)
        v, r = run_sparql_on_graph(g, fallback)
        return {"query": fallback, "vars": v, "rows": r, "repaired": True, "error": None}
    except:
        return {"query": sparql, "vars": [], "rows": [], "repaired": True, "error": str(err)}


def clean_uri_for_display(cell):
    # Nettoie une URI pour l'affichage humain.
    NAMESPACES = [
        (ENT_NS, ""), (OCT_NS, ""),
        (WDT_NS, "wdt:"), (WD_NS, "wd:"),
        ("http://www.w3.org/2000/01/rdf-schema#", "rdfs:"),
        ("http://www.w3.org/1999/02/22-rdf-syntax-ns#", "rdf:"),
        ("http://www.w3.org/2002/07/owl#", "owl:"),
    ]
    for ns, prefix in NAMESPACES:
        if cell.startswith(ns):
            cell = prefix + cell[len(ns):]
            break
    cell = cell.replace("_", " ")
    try:
        cell = unquote(cell)
    except:
        pass
    return cell.strip()


def format_results(result):
    # Formate les résultats SPARQL pour l'affichage.
    if result.get("error"):
        return f"❌ Erreur : {result['error'][:200]}"
    
    rows = result.get("rows", [])
    if not rows:
        return "Aucun résultat trouvé dans la KB."
    
    headers = result.get("vars", [])
    cleaned_rows = []
    for row in rows[:MAX_RESULTS_SHOW]:
        cleaned_rows.append([clean_uri_for_display(c) for c in row])
    
    lines = []
    if headers:
        lines.append(" | ".join(headers))
        lines.append("-" * len(lines[0]))
    for row in cleaned_rows:
        lines.append(" | ".join(row))
    
    text = "\n".join(lines)
    if len(rows) > MAX_RESULTS_SHOW:
        text += f"\n... ({len(rows)} résultats au total)"
    return text


print("\n✓ Pipeline RAG v3 défini (templates + mistral:7b)")

# Test rapide
for test_q in ["Quelles espèces de poulpes vivent dans le Pacifique ?",
               "Quel est le parent taxonomique d'Octopus vulgaris ?"]:
    print(f"\nTest : '{test_q}'")
    test_sparql = generate_sparql(test_q, schema_summary)
    print(test_sparql[:300])


## Etape 6 : Évaluation Baseline vs RAG

In [ ]:
# ÉTAPE 6 — ÉVALUATION BASELINE vs SPARQL-RAG

print("\n" + "=" * 70)
print("  ÉVALUATION — Baseline vs SPARQL-RAG")
print("=" * 70)

evaluation_rows = []
for q in EVAL_QUESTIONS:
    print(f"\n{'='*60}\n❓ {q}")
    baseline_ans = baseline_results.get(q, answer_no_rag(q))
    result       = answer_with_sparql_rag(g_rag, schema_summary, q, try_repair=True)
    rag_text     = format_results(result)
    print(f"  Baseline : {baseline_ans[:150]}")
    print(f"  RAG      : {rag_text[:150]}")
    evaluation_rows.append({
        "Question":    q,
        "Baseline":    baseline_ans[:150] + "..." if len(baseline_ans) > 150 else baseline_ans,
        "RAG SPARQL":  rag_text[:150]     + "..." if len(rag_text)     > 150 else rag_text,
        "Nb résultats":len(result.get("rows", [])),
        "Réparation":  result["repaired"],
        "Erreur":      bool(result.get("error")),
    })

eval_rag_df = pd.DataFrame(evaluation_rows)
print("\n" + "=" * 70)
print(eval_rag_df[["Question", "Nb résultats", "Réparation", "Erreur"]].to_string(index=False))
n_ok  = sum(1 for r in evaluation_rows if not r["Erreur"] and r["Nb résultats"] > 0)
n_rep = sum(1 for r in evaluation_rows if r["Réparation"])
print(f"\nSuccès : {n_ok}/{len(evaluation_rows)} | Self-repair : {n_rep}")
eval_rag_df.to_csv("lab4_evaluation.csv", index=False, encoding="utf-8-sig")
print("→ lab4_evaluation.csv")


## Etape 7 : Interface Gradio

In [ ]:
# ÉTAPE 7 — INTERFACE GRADIO (compatible Gradio v4+)

def synthesize_answer(question, sparql_results_text, model=OLLAMA_MODEL):
    if "Aucun résultat" in sparql_results_text or "Erreur" in sparql_results_text:
        return sparql_results_text
    prompt = (
        "Tu es un expert en biologie marine spécialisé dans les poulpes.\n"
        "Voici les données extraites d'un graphe de connaissances :\n\n"
        f"{sparql_results_text[:1500]}\n\n"
        f"Question de l'utilisateur : {question}\n\n"
        "Réponds en français de manière claire et concise en te basant "
        "UNIQUEMENT sur les données ci-dessus."
    )
    answer = ask_local_llm(prompt, model=model)
    return answer if answer and not answer.startswith("[ERREUR]") else sparql_results_text

try:

    def chat_rag(question, model, show_sparql, history):
        if not question.strip():
            return "", "", "", history
        baseline_ans = answer_no_rag(question, model=model)
        result = answer_with_sparql_rag(g_rag, schema_summary, question,
                                         try_repair=True, model=model)
        rag_raw = format_results(result)
        if result.get("rows"):
            rag_answer = synthesize_answer(question, rag_raw, model=model)
            status = "⚠ Réparé" if result["repaired"] else "✓"
            n_results = len(result.get("rows", []))
            rag_display = f"{status} | {n_results} résultat(s)\n\n{rag_answer}"
        else:
            rag_display = rag_raw
        history.append({"role": "user", "content": question})
        history.append({"role": "assistant", "content": rag_display})
        sparql_display = result["query"] if show_sparql else "(désactivé)"
        return baseline_ans, rag_display, sparql_display, history

    with gr.Blocks(title="🐙 Chatbot Poulpes", theme=gr.themes.Soft(primary_hue="blue")) as demo:
        gr.Markdown(
            "# 🐙 Chatbot Poulpes — RAG avec Graphe de Connaissances\n"
            "**Auteurs :** Bilal Gougis & Chadi Al Kerdi | **Cours :** Web Mining & Sémantique\n\n"
            "Posez une question sur les poulpes : le système génère une requête SPARQL, "
            "interroge la KB `kb_expanded.ttl` et retourne une réponse ancrée dans les données."
        )
        with gr.Row():
            with gr.Column(scale=3):
                q_in = gr.Textbox(
                    label="Votre question",
                    placeholder="Ex: Quel est le parent taxonomique d'Octopus vulgaris ?",
                    lines=2)
                with gr.Row():
                    mdl = gr.Dropdown(
                        choices=["mistral:7b", "llama3.2:3b", "llama3.2:1b", "gemma:2b"],
                        value=OLLAMA_MODEL, label="Modèle Ollama")
                    sparql_cb = gr.Checkbox(value=True, label="Afficher SPARQL")
                with gr.Row():
                    submit = gr.Button("🔍 Interroger", variant="primary")
                    clear = gr.Button("🗑 Effacer")
            with gr.Column(scale=1):
                gr.Markdown("### Exemples de questions")
                for eq in EVAL_QUESTIONS:
                    gr.Button(eq, size="sm").click(fn=lambda x=eq: x, outputs=q_in)
        with gr.Row():
            base_out = gr.Textbox(label="🤖 Baseline — LLM seul", lines=8, interactive=False)
            rag_out = gr.Textbox(label="🕸️ SPARQL-RAG — LLM + Graphe", lines=8, interactive=False)
        sparql_out = gr.Code(label="Requête SPARQL générée", language="sql", lines=8)
        chat_out = gr.Chatbot(label="Historique", height=250, type="messages")
        hist_state = gr.State([])
        for trigger in [submit.click, q_in.submit]:
            trigger(
                fn=chat_rag,
                inputs=[q_in, mdl, sparql_cb, hist_state],
                outputs=[base_out, rag_out, sparql_out, hist_state]
            ).then(fn=lambda h: h, inputs=hist_state, outputs=chat_out)
        clear.click(
            fn=lambda: ([], [], "", "", ""),
            outputs=[hist_state, chat_out, base_out, rag_out, sparql_out]
        )
        gr.Markdown(
            "---\n**Source KB** : `kb_expanded.ttl` | "
            "**Modèle** : LLM local via [Ollama](https://ollama.com/)"
        )
    demo.launch(share=False, inbrowser=True, quiet=True)

except ImportError:
    print("⚠ Gradio non installé : pip install gradio")


## Démo CLI

In [ ]:
# DÉMO CLI — Interface en ligne de commande

def pretty_print_result(result):
    # Affiche les résultats SPARQL de manière lisible.
    if result.get("error"):
        print(f"\n  [Erreur] {result['error'][:200]}")
    print(f"\n  [Requête SPARQL]")
    print(f"  {result['query'][:300]}")
    print(f"\n  [Réparé ?] {result['repaired']}")
    
    rows = result.get("rows", [])
    if not rows:
        print("\n  [Aucun résultat]")
        return
    
    print(f"\n  [Résultats : {len(rows)} lignes]")
    formatted = format_results(result)
    for line in formatted.split("\n")[:25]:
        print(f"  {line}")

def cli_demo():
    # Boucle interactive CLI pour tester le chatbot.
    print("\n" + "=" * 60)
    print("  🐙 CHATBOT POULPES — Démo CLI")
    print("  Tapez 'quit' pour quitter")
    print("=" * 60)
    
    while True:
        try:
            q = input("\n❓ Question : ").strip()
        except (EOFError, KeyboardInterrupt):
            break
        if q.lower() in ("quit", "exit", "q"):
            break
        if not q:
            continue
        
        print("\n--- Baseline (LLM seul) ---")
        baseline = answer_no_rag(q)
        print(f"  {baseline[:300]}")
        
        print("\n--- SPARQL-RAG (LLM + Graphe) ---")
        result = answer_with_sparql_rag(g_rag, schema_summary, q, try_repair=True)
        pretty_print_result(result)
        
        # Synthèse en langage naturel
        if result.get("rows"):
            rag_text = format_results(result)
            print("\n--- Réponse synthétisée ---")
            synth = synthesize_answer(q, rag_text)
            print(f"  {synth[:400]}")
    
    print("\n✓ Fin de la démo CLI")

# Décommenter pour lancer la démo CLI :
# cli_demo()

# Démonstration automatique avec les questions d'évaluation
print("\n" + "=" * 60)
print("  DÉMO CLI — Exemples automatiques")
print("=" * 60)
for q in EVAL_QUESTIONS[:2]:
    print(f"\n{'='*50}")
    print(f"❓ {q}")
    print("\n--- Baseline ---")
    print(f"  {answer_no_rag(q)[:200]}")
    print("\n--- SPARQL-RAG ---")
    result = answer_with_sparql_rag(g_rag, schema_summary, q, try_repair=True)
    pretty_print_result(result)


---
Lab 4 terminé.